# **Replica pipeline**

### **Funzioni comuni**

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os

DATA_DIR = "VitaStress/data/"
WINDOW = "30s"  # "resampled to have a sampling frequency of 1/30 seconds" (preprint, sez. 3.6)
OUT_PATH = "analysis/windows_30s.csv"  # tabella delle finestre (non normalizzata)
OUT_PATH_ZSCORE = "analysis/windows_30s_normalized.csv"
OUT_PATH_ZSCORE_FILTERED = "analysis/windows_30s_normalized_filtered.csv"
OUT_PATH_START = "analysis/windows_30s_start.csv"  # finestre di bordo incluse, baseline tagliata (non normalizzata)

# baseline più lunghe di BASELINE_MAX vengono ridotte agli ultimi BASELINE_KEEP prima di
# "Baseline Stop" (protocollo: ~10 min; tutte 9.8-10.2 min tranne id_a5e823ad, vedi label_windows)
BASELINE_MAX = pd.Timedelta("12min")
BASELINE_KEEP = pd.Timedelta("10min")

# etichette Start/Stop per fase, osservate in *_annotation.csv (non uniformi tra loro,
# "public" usa deliberatamente solo il discorso, non la preparazione)
PHASE_LABELS = {
    "baseline": ("Baseline Start (Start of Experiment)", "Baseline Stop"),
    "cognitive": ("Cognitive: Start", "Cognitive: Stop"),
    "physical": ("Physical: Start", "Physical: Stop"),
    "public": ("Public Speaking Start", "Public Speaking Stop"),
}
PHASES = list(PHASE_LABELS.keys())  # solo i nomi delle fasi: baseline, cognitive, physical, public

# schema della pipeline (preprint, Figura 3): l'accelerometro (grezzo, ~32 Hz) viene
# AGGREGATO per finestra con mean/std/var, i segnali vitali (già processati da Corsano)
# vengono solo RICAMPIONATI sulla stessa griglia
ACC_SOURCE = ("_acc.csv", ["accX", "accY", "accZ"])
VITAL_SOURCES = {
    "_activity.csv": ["bpm"],
    "_emography.csv": ["stress_skin"],  # EDA / skin conductance
    "_temperature.csv": ["temp_sk1"],  # la "skin temperature" del paper (in realtà CBT); temp_sk2 è sempre 0
}

# colonne in cui un valore mancante è scritto come 0 invece che lasciato vuoto
ZERO_IS_MISSING = {"bpm", "temp_sk1"}

def subject_files(subject_dir):
    """Path di tutti i file del soggetto, indicizzati per suffisso esatto:
    "<uuid>_temperature.csv" -> "_temperature.csv". Gli UUID non contengono "_",
    quindi il suffisso inizia al primo "_". Con endswith() invece
    "_heat_flux_sensor_temperature.csv" verrebbe preso anche per "_temperature.csv"."""
    with os.scandir(subject_dir) as files:
        return {f.name[f.name.find("_"):]: f.path for f in files}


def get_phase_intervals(annotation_csv_path):
    """Per ciascuna fase: (ultimo Start prima dell'ultimo Stop, ultimo Stop).
    Gestisce riavvii/duplicati nelle annotation. Ritorna None per una fase
    se manca lo Start o lo Stop corrispondente."""
    ann = pd.read_csv(annotation_csv_path)
    # format="ISO8601": alcune righe non hanno i microsecondi, un formato fisso
    # inferito dalla prima riga fallisce sulle righe con precisione diversa
    ann["timestamp"] = pd.to_datetime(ann["timestamp"], utc=True, format="ISO8601")

    intervals = {}
    for phase, (start_label, stop_label) in PHASE_LABELS.items():
        stops = ann.loc[ann["Button Name"] == stop_label, "timestamp"]
        if stops.empty:
            intervals[phase] = None
            continue
        stop = stops.max()

        starts = ann.loc[(ann["Button Name"] == start_label) & (ann["timestamp"] < stop), "timestamp"]
        if starts.empty:
            intervals[phase] = None
            continue
        start = starts.max()

        intervals[phase] = (start, stop)
    return intervals


def read_signal(path, columns):
    """Colonne richieste di un CSV, indicizzate per data, con gli zeri-mancanti a NaN."""
    df = pd.read_csv(path, usecols=["date", *columns])
    df["date"] = pd.to_datetime(df["date"], utc=True, format="ISO8601")
    for col in columns:
        if col in ZERO_IS_MISSING:
            df[col] = df[col].replace(0, np.nan)
    return df.set_index("date")


def resample_vital(signal):
    """Ricampiona un segnale vitale sulla griglia da WINDOW. Se il segnale è più lento
    della finestra (emography: 1 valore/min) il ricampionamento lascia finestre vuote
    tra un valore e l'altro: riempiamo solo quei buchi per interpolazione lineare, non i
    mancanti veri (nel paper stress_skin ha ~0% di mancanti, Figura 4)."""
    resampled = signal.resample(WINDOW).mean()
    native_step = signal.index.to_series().diff().median()
    holes = int(native_step / pd.Timedelta(WINDOW)) - 1
    if holes > 0:
        resampled = resampled.interpolate(method="linear", limit=holes, limit_area="inside")
    return resampled


def subject_windows(paths):
    """Una riga per finestra da WINDOW con tutte le feature del soggetto affiancate
    (allineamento: tutte le sorgenti sono sulla stessa griglia temporale)."""
    suffix, columns = ACC_SOURCE
    acc = read_signal(paths[suffix], columns).resample(WINDOW).agg(["mean", "std", "var"])
    acc.columns = [f"{col}_{stat}" for col, stat in acc.columns]  # ("accX", "mean") -> "accX_mean"

    signals = [acc]
    for suffix, columns in VITAL_SOURCES.items():
        signals.append(resample_vital(read_signal(paths[suffix], columns)))
    print("START SUBJECT WINDOW")
    print(pd.concat(signals, axis=1, sort=True))
    print("END SUBJECT WINDOW")
    return pd.concat(signals, axis=1, sort=True)


def subject_windows_zscore(paths):
    """Come subject_windows, ma ogni feature è già z-scorata per soggetto usando TUTTA
    la registrazione (anche briefing, preparazione, pause, recovery), non solo le 4 fasi.
    Le statistiche si calcolano prima di label_windows, che poi tiene solo le finestre
    nelle fasi."""
    windows = subject_windows(paths)
    return (windows - windows.mean()) / windows.std(ddof=0)


def experiment_span(annotation_csv_path):
    """(prima annotation, ultima annotation) del soggetto. Le righe senza timestamp
    (es. l'ultimo "Cognitive Stop" di id_7bb4dafd) vengono ignorate."""
    ann = pd.read_csv(annotation_csv_path)
    timestamps = pd.to_datetime(ann["timestamp"], utc=True, format="ISO8601").dropna()
    return timestamps.min(), timestamps.max()


def subject_windows_zscore_filtered(paths):
    """Come subject_windows_zscore, ma media e std di ogni feature si calcolano solo
    sulle finestre interamente comprese tra la prima e l'ultima annotation, escludendo
    quello che è stato registrato prima e dopo l'esperimento. Il risultato contiene
    comunque tutte le finestre: è label_windows a tenere poi solo quelle nelle fasi."""
    windows = subject_windows(paths)
    start, stop = experiment_span(paths["_annotation.csv"])
    inside = (windows.index >= start) & (windows.index + pd.Timedelta(WINDOW) <= stop)
    experiment = windows[inside]
    return (windows - experiment.mean()) / experiment.std(ddof=0)


def label_windows(windows, intervals, rule="full", trim_baseline=False):
    """Tiene solo le finestre che appartengono a una delle 4 fasi e aggiunge la colonna
    "phase". L'indice è l'inizio della finestra. La griglia da 30 s è allineata
    all'orologio, gli Start/Stop no: rule decide cosa fare con le finestre a cavallo
    del confine di una fase.
    - "full":  solo finestre interamente dentro la fase (le finestre di bordo si scartano)
    - "start": finestre che INIZIANO dentro la fase, cioè l'etichetta che riceve ogni
               riga del resample in base al suo timestamp. Con questa regola i campioni di
               stress tornano con la Figura 8 del preprint (774 contro 757 del paper,
               che probabilmente ha scartato anche le righe con NaN)
    Con trim_baseline=True una baseline più lunga di BASELINE_MAX viene ridotta agli
    ultimi BASELINE_KEEP: in id_a5e823ad "Baseline Start" è premuto ~1 h prima
    dell'inizio della registrazione."""
    labeled = []
    for phase, interval in intervals.items():
        if interval is None:
            continue
        start, stop = interval
        if trim_baseline and phase == "baseline" and stop - start > BASELINE_MAX:
            start = stop - BASELINE_KEEP

        if rule == "full":
            inside = (windows.index >= start) & (windows.index + pd.Timedelta(WINDOW) <= stop)
        elif rule == "start":
            inside = (windows.index >= start) & (windows.index < stop)
        else:
            raise ValueError(f"rule sconosciuta: {rule}")
        labeled.append(windows[inside].assign(phase=phase))
    return pd.concat(labeled)


def build_window_table(windows_fn=subject_windows, label_rule="full", trim_baseline=False):
    """Tabella di tutti i soggetti: una riga per finestra etichettata, colonne
    feature + "phase" + "subject". Le feature sono quelle restituite da windows_fn
    (subject_windows: NON normalizzate); label_rule e trim_baseline vanno a
    label_windows. Ritorna anche la lista (soggetto, motivo) di cosa è stato scartato."""
    tables = []
    skipped = []

    with os.scandir(DATA_DIR) as entries:
        subject_dirs = sorted((e for e in entries if e.is_dir()), key=lambda e: e.name)

    for subject_dir in subject_dirs:
        paths = subject_files(subject_dir)

        missing = [s for s in ["_annotation.csv", ACC_SOURCE[0], *VITAL_SOURCES] if s not in paths]
        if missing:
            skipped.append((subject_dir.name, f"file mancanti: {missing}"))
            continue

        intervals = get_phase_intervals(paths["_annotation.csv"])
        for phase, interval in intervals.items():
            if interval is None:
                skipped.append((subject_dir.name, f"fase {phase} senza Start/Stop"))

        print("INTERVALS: %s", intervals)
        windows = label_windows(windows_fn(paths), intervals, label_rule, trim_baseline)
        print("WINDOWS: %s", windows)
        tables.append(windows.assign(subject=subject_dir.name))

    return pd.concat(tables), skipped


def build_window_table_zscore():
    """Come build_window_table, ma con le feature z-scorate su tutta la registrazione
    di ciascun soggetto (vedi subject_windows_zscore)."""
    return build_window_table(subject_windows_zscore)


def build_window_table_zscore_filtered():
    """Come build_window_table, ma con le feature z-scorate sull'intervallo tra prima
    e ultima annotation di ciascun soggetto (vedi subject_windows_zscore_filtered)."""
    return build_window_table(subject_windows_zscore_filtered)


### **Estrazione dati da VitaStress**

Passi 1–4 della Figura 3 del preprint: estrazione feature (`bpm` da `*_activity.csv`, `stress_skin` da `*_emography.csv`, `temp_sk1` da `*_temperature.csv`, `accX/Y/Z` da `*_acc.csv`), ricampionamento dei segnali vitali a 1 valore ogni 30 s (buchi di `stress_skin`, che è a 1/min, riempiti per interpolazione lineare), aggregazione dell'accelerometro (mean/std/var) sulle stesse finestre, allineamento ed etichette dalle annotation.  
La tabella è salvata **non normalizzata** in `VitaStress/analysis/windows_30s.pkl`; per riusarla: `table = pd.read_pickle("VitaStress/analysis/windows_30s.pkl")`, poi `zscore_per_subject`.

In [2]:
# Passi 1-4 della pipeline (preprint, Figura 3): estrazione feature, ricampionamento /
# aggregazione a 30 s, allineamento, etichette. La tabella viene salvata PRIMA della
# normalizzazione per soggetto, così si può cambiare quella senza rileggere i CSV.
import os
import pandas as pd

table, skipped = build_window_table()

# l'inizio della finestra diventa una colonna normale; "phase" tiene le 4 fasi originali,
# le etichette binaria/ternaria si ricavano dopo con una mappatura
table = table.rename_axis("window_start").reset_index()
feature_columns = [col for col in table.columns if col not in ("subject", "phase", "window_start")]
table = table[["subject", "phase", "window_start", *feature_columns]]

os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)
table.to_csv(OUT_PATH)
# table.to_pickle(OUT_PATH) # per versione in .pkl


if skipped:
    print("Scartati:")
    for name, reason in skipped:
        print(f"  - {name}: {reason}")
print(f"salvata {OUT_PATH}: {table.shape[0]} finestre x {len(feature_columns)} feature")
print("soggetti:", table["subject"].nunique())
print("finestre per fase:", table.groupby("phase").size().to_dict())
# confronto con la Figura 4 del preprint (% mancanti per classe)
print("\n% mancanti per fase:")
print((table.groupby("phase")[feature_columns].apply(lambda g: g.isna().mean()) * 100).round(1).T)

INTERVALS: %s {'baseline': (Timestamp('2035-03-15 15:12:28.061472+0000', tz='UTC'), Timestamp('2035-03-15 15:22:29.092427+0000', tz='UTC')), 'cognitive': (Timestamp('2035-03-15 15:26:09.148608+0000', tz='UTC'), Timestamp('2035-03-15 15:31:09.653759+0000', tz='UTC')), 'physical': (Timestamp('2035-03-15 15:39:59.855568+0000', tz='UTC'), Timestamp('2035-03-15 15:48:39.323104+0000', tz='UTC')), 'public': (Timestamp('2035-03-15 16:04:35.044096+0000', tz='UTC'), Timestamp('2035-03-15 16:09:34.702965+0000', tz='UTC'))}
START SUBJECT WINDOW
                            accX_mean    accX_std      accX_var  accY_mean  \
date                                                                         
2035-03-15 14:50:00+00:00   29.946875    0.508828      0.258906 -25.417708   
2035-03-15 14:50:30+00:00   30.228125    0.515698      0.265944 -25.081250   
2035-03-15 14:51:00+00:00   30.262500    0.524529      0.275130 -24.931250   
2035-03-15 14:51:30+00:00   30.254167    0.493950      0.243987 -24.791

### **Estrazione dati da VitaStress**
Versione con z-score calcolato su tutta la registrazione

In [2]:
import os
import pandas as pd

table, skipped = build_window_table_zscore()

# l'inizio della finestra diventa una colonna normale; "phase" tiene le 4 fasi originali,
# le etichette binaria/ternaria si ricavano dopo con una mappatura
table = table.rename_axis("window_start").reset_index()
feature_columns = [col for col in table.columns if col not in ("subject", "phase", "window_start")]
table = table[["subject", "phase", "window_start", *feature_columns]]

os.makedirs(os.path.dirname(OUT_PATH_ZSCORE), exist_ok=True)
table.to_csv(OUT_PATH_ZSCORE)
# table.to_pickle(OUT_PATH) # per versione in .pkl


if skipped:
    print("Scartati:")
    for name, reason in skipped:
        print(f"  - {name}: {reason}")
print(f"salvata {OUT_PATH_ZSCORE}: {table.shape[0]} finestre x {len(feature_columns)} feature")
print("soggetti:", table["subject"].nunique())
print("finestre per fase:", table.groupby("phase").size().to_dict())
# confronto con la Figura 4 del preprint (% mancanti per classe)
print("\n% mancanti per fase:")
print((table.groupby("phase")[feature_columns].apply(lambda g: g.isna().mean()) * 100).round(1).T)

NameError: name 'build_window_table_zscore' is not defined

### **Estrazione dati da VitaStress**
Versione con z-score calcolato dalla prima all'ultima annotation (esclusi i tratti registrati prima e dopo l'esperimento)

In [47]:
import os
import pandas as pd

table, skipped = build_window_table_zscore_filtered()

# l'inizio della finestra diventa una colonna normale; "phase" tiene le 4 fasi originali,
# le etichette binaria/ternaria si ricavano dopo con una mappatura
table = table.rename_axis("window_start").reset_index()
feature_columns = [col for col in table.columns if col not in ("subject", "phase", "window_start")]
table = table[["subject", "phase", "window_start", *feature_columns]]

os.makedirs(os.path.dirname(OUT_PATH_ZSCORE_FILTERED), exist_ok=True)
table.to_csv(OUT_PATH_ZSCORE_FILTERED)
# table.to_pickle(OUT_PATH) # per versione in .pkl


if skipped:
    print("Scartati:")
    for name, reason in skipped:
        print(f"  - {name}: {reason}")
print(f"salvata {OUT_PATH_ZSCORE_FILTERED}: {table.shape[0]} finestre x {len(feature_columns)} feature")
print("soggetti:", table["subject"].nunique())
print("finestre per fase:", table.groupby("phase").size().to_dict())
# confronto con la Figura 4 del preprint (% mancanti per classe)
print("\n% mancanti per fase:")
print((table.groupby("phase")[feature_columns].apply(lambda g: g.isna().mean()) * 100).round(1).T)

INTERVALS: %s {'baseline': (Timestamp('2035-03-15 15:12:28.061472+0000', tz='UTC'), Timestamp('2035-03-15 15:22:29.092427+0000', tz='UTC')), 'cognitive': (Timestamp('2035-03-15 15:26:09.148608+0000', tz='UTC'), Timestamp('2035-03-15 15:31:09.653759+0000', tz='UTC')), 'physical': (Timestamp('2035-03-15 15:39:59.855568+0000', tz='UTC'), Timestamp('2035-03-15 15:48:39.323104+0000', tz='UTC')), 'public': (Timestamp('2035-03-15 16:04:35.044096+0000', tz='UTC'), Timestamp('2035-03-15 16:09:34.702965+0000', tz='UTC'))}
START SUBJECT WINDOW
                            accX_mean    accX_std      accX_var  accY_mean  \
date                                                                         
2035-03-15 14:50:00+00:00   29.946875    0.508828      0.258906 -25.417708   
2035-03-15 14:50:30+00:00   30.228125    0.515698      0.265944 -25.081250   
2035-03-15 14:51:00+00:00   30.262500    0.524529      0.275130 -24.931250   
2035-03-15 14:51:30+00:00   30.254167    0.493950      0.243987 -24.791

### **Estrazione dati da VitaStress**
Versione con le finestre a cavallo dei confini di fase incluse (tengo quelle che *iniziano* dentro la fase) e baseline di `id_a5e823ad` ridotta agli ultimi 10 min. Tabella non normalizzata.

In [48]:
# Come la tabella grezza, ma con le finestre di bordo incluse (label_rule="start") e la
# baseline troppo lunga di id_a5e823ad ridotta agli ultimi 10 min
import os
import pandas as pd

table, skipped = build_window_table(label_rule="start", trim_baseline=True)

# l'inizio della finestra diventa una colonna normale; "phase" tiene le 4 fasi originali,
# le etichette binaria/ternaria si ricavano dopo con una mappatura
table = table.rename_axis("window_start").reset_index()
feature_columns = [col for col in table.columns if col not in ("subject", "phase", "window_start")]
table = table[["subject", "phase", "window_start", *feature_columns]]

os.makedirs(os.path.dirname(OUT_PATH_START), exist_ok=True)
table.to_csv(OUT_PATH_START)


if skipped:
    print("Scartati:")
    for name, reason in skipped:
        print(f"  - {name}: {reason}")
print(f"salvata {OUT_PATH_START}: {table.shape[0]} finestre x {len(feature_columns)} feature")
print("soggetti:", table["subject"].nunique())
print("finestre per fase:", table.groupby("phase").size().to_dict())
# confronto con la Figura 4 del preprint (% mancanti per classe)
print("\n% mancanti per fase:")
print((table.groupby("phase")[feature_columns].apply(lambda g: g.isna().mean()) * 100).round(1).T)

# confronto con la Figura 8 del preprint (binario): 357 neutral, 757 stress
complete = table[feature_columns].notna().all(axis=1)
is_baseline = table["phase"] == "baseline"
print(f"\nbaseline / stress: {is_baseline.sum()} / {(~is_baseline).sum()} "
      f"(senza NaN: {(is_baseline & complete).sum()} / {(~is_baseline & complete).sum()}; paper: 357 / 757)")


INTERVALS: %s {'baseline': (Timestamp('2035-03-15 15:12:28.061472+0000', tz='UTC'), Timestamp('2035-03-15 15:22:29.092427+0000', tz='UTC')), 'cognitive': (Timestamp('2035-03-15 15:26:09.148608+0000', tz='UTC'), Timestamp('2035-03-15 15:31:09.653759+0000', tz='UTC')), 'physical': (Timestamp('2035-03-15 15:39:59.855568+0000', tz='UTC'), Timestamp('2035-03-15 15:48:39.323104+0000', tz='UTC')), 'public': (Timestamp('2035-03-15 16:04:35.044096+0000', tz='UTC'), Timestamp('2035-03-15 16:09:34.702965+0000', tz='UTC'))}
START SUBJECT WINDOW
                            accX_mean    accX_std      accX_var  accY_mean  \
date                                                                         
2035-03-15 14:50:00+00:00   29.946875    0.508828      0.258906 -25.417708   
2035-03-15 14:50:30+00:00   30.228125    0.515698      0.265944 -25.081250   
2035-03-15 14:51:00+00:00   30.262500    0.524529      0.275130 -24.931250   
2035-03-15 14:51:30+00:00   30.254167    0.493950      0.243987 -24.791

## ***Classifiers***

In [49]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier

# insiemi di feature del paper (sez. 5: "All", "Physio", "Motion")
PHYSIO_FEATURES = ["bpm", "stress_skin", "temp_sk1"]
MOTION_FEATURES = [f"acc{axis}_{stat}" for axis in "XYZ" for stat in ("mean", "std", "var")]
FEATURE_SETS = {
    "All": PHYSIO_FEATURES + MOTION_FEATURES,
    "Physio": PHYSIO_FEATURES,
    "Motion": MOTION_FEATURES,
}

# fase originale -> classe, per i due compiti del paper (sez. 5)
TASKS = {
    "binary": {"baseline": "baseline", "physical": "stress", "cognitive": "stress", "public": "stress"},
    "ternary": {"baseline": "baseline", "physical": "physical",
                "cognitive": "cognitive+public", "public": "cognitive+public"},
}


def zscore_per_subject(table, columns):
    """z-score di ogni colonna separatamente per ciascun soggetto, calcolato sulle
    righe delle 4 fasi ("normalized to account for inter-personal differences")."""
    by_subject = table.groupby("subject")[columns]
    normalized = table.copy()
    normalized[columns] = (table[columns] - by_subject.transform("mean")) / by_subject.transform("std", ddof=0)
    return normalized


def get_classifiers():
    # i 6 classificatori del paper (sez. 3.6); iperparametri fissi, il paper fa grid search
    return {
        'RF': RandomForestClassifier(n_estimators=100, random_state=42),
        'LDA': LinearDiscriminantAnalysis(),
        'kNN (k=9)': KNeighborsClassifier(n_neighbors=9),  # nel paper WESAD viene utilizzato k=9
        'DT': DecisionTreeClassifier(criterion='gini', max_depth=5, min_samples_leaf=5, random_state=42),
        'LR': LogisticRegression(max_iter=1000, C=1.0),
        'XGB': XGBClassifier(random_state=42),
    }


def evaluate_loso(X, y, groups, class_names):
    """Leave-one-subject-out: un fold per soggetto. Accuracy e macro-F1 sono calcolate
    per fold e poi mediate (media ± std sui fold, come nelle Tabelle 2-5 del paper);
    la matrice di confusione somma le predizioni di tutti i fold."""
    results = {}
    print(f'{"":<10} {"acc":>13} {"macro-F1":>13}')

    for name, clf in get_classifiers().items():
        accs, f1s = [], []
        y_true_all, y_pred_all = [], []

        for tr, te in LeaveOneGroupOut().split(X, y, groups=groups):
            clf.fit(X[tr], y[tr])
            pred = clf.predict(X[te])

            accs.append(accuracy_score(y[te], pred))
            f1s.append(f1_score(y[te], pred, average='macro', zero_division=0))
            y_true_all.extend(y[te])
            y_pred_all.extend(pred)

        results[name] = {
            'acc': np.mean(accs), 'acc_std': np.std(accs),
            'f1': np.mean(f1s), 'f1_std': np.std(f1s),
            'cm': confusion_matrix(y_true_all, y_pred_all, labels=range(len(class_names))),
        }
        print(f'{name:<10} {np.mean(accs):>6.2f} ± {np.std(accs):.2f} {np.mean(f1s):>6.2f} ± {np.std(f1s):.2f}')

    best = max(results, key=lambda k: results[k]['f1'])
    print(f'Migliore (F1): {best}  F1={results[best]["f1"]:.2f} ± {results[best]["f1_std"]:.2f}')
    print('matrice di confusione (righe = vero, colonne = predetto):', list(class_names))
    print(results[best]['cm'])
    return results, best


# base dello z-score per soggetto:
# "phases"    -> tabella grezza, z-score calcolato qui sulle sole finestre delle 4 fasi
# "recording" -> tabella già z-scorata su tutta la registrazione (build_window_table_zscore)
# "experiment" -> tabella già z-scorata tra prima e ultima annotation (build_window_table_zscore_filtered)
ZSCORE_BASE = "phases"
# tabella grezza usata con "phases": OUT_PATH (solo finestre intere) oppure OUT_PATH_START
RAW_PATH = OUT_PATH

feature_columns = FEATURE_SETS["All"]
if ZSCORE_BASE == "phases":
    table = pd.read_csv(RAW_PATH, parse_dates=["window_start"])
    table_z = zscore_per_subject(table, feature_columns)
elif ZSCORE_BASE == "recording":
    table_z = pd.read_csv(OUT_PATH_ZSCORE, parse_dates=["window_start"])
else:
    table_z = pd.read_csv(OUT_PATH_ZSCORE_FILTERED, parse_dates=["window_start"])

# LDA, kNN e LR non accettano NaN: si scartano le finestre con almeno una feature mancante
complete = table_z[feature_columns].notna().all(axis=1)
print(f"finestre scartate per NaN: {(~complete).sum()} su {len(table_z)}")
table_z = table_z[complete].reset_index(drop=True)
groups = table_z["subject"].to_numpy()

all_results = {}
for task, mapping in TASKS.items():
    # classi -> interi 0..n-1 (XGBoost li richiede); class_names[i] è il nome della classe i
    y, class_names = pd.factorize(table_z["phase"].map(mapping), sort=True)
    for set_name, columns in FEATURE_SETS.items():
        print(f"\n=== {task} / {set_name} ({len(columns)} feature)")
        X = table_z[columns].to_numpy()
        all_results[(task, set_name)], _ = evaluate_loso(X, y, groups, class_names)

# riepilogo nello stesso formato delle Tabelle 2-5 del paper (righe = feature set, colonne = classificatori)
for task in TASKS:
    for metric in ("acc", "f1"):
        summary = pd.DataFrame({set_name: {clf: r[metric] for clf, r in all_results[(task, set_name)].items()}
                                for set_name in FEATURE_SETS}).T
        print(f"\n{task} - {metric}")
        print(summary.round(2))

finestre scartate per NaN: 28 su 1147

=== binary / All (12 feature)
                     acc      macro-F1
RF           0.92 ± 0.12   0.92 ± 0.15
LDA          0.91 ± 0.14   0.90 ± 0.15
kNN (k=9)    0.88 ± 0.15   0.88 ± 0.16
DT           0.86 ± 0.16   0.82 ± 0.23
LR           0.92 ± 0.12   0.91 ± 0.14
XGB          0.92 ± 0.13   0.91 ± 0.15
Migliore (F1): RF  F1=0.92 ± 0.15
matrice di confusione (righe = vero, colonne = predetto): ['baseline', 'stress']
[[371  59]
 [ 31 658]]

=== binary / Physio (3 feature)
                     acc      macro-F1
RF           0.91 ± 0.11   0.90 ± 0.13
LDA          0.90 ± 0.07   0.90 ± 0.07
kNN (k=9)    0.90 ± 0.12   0.90 ± 0.13
DT           0.93 ± 0.09   0.92 ± 0.11
LR           0.92 ± 0.08   0.92 ± 0.08
XGB          0.88 ± 0.11   0.88 ± 0.12
Migliore (F1): DT  F1=0.92 ± 0.11
matrice di confusione (righe = vero, colonne = predetto): ['baseline', 'stress']
[[370  60]
 [ 29 660]]

=== binary / Motion (9 feature)
                     acc      macro-F1
RF  

# Complete pipeline

In [1]:
"""Replica della pipeline del preprint VitaStress (Figura 3) e della classificazione (sez. 5)."""
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import GridSearchCV, LeaveOneGroupOut
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier

ROOT = Path.cwd().parent  # il notebook gira in code/ -> cartella Tirocinio/
DATA_DIR = ROOT / "VitaStress" / "data"
OUT_DIR = ROOT / "analysis"

# --- scelte della pipeline (il paper non le specifica)
# finestre a cavallo della fine di una fase:
#   "full"  -> scartate, si tengono solo le finestre interamente dentro la fase
#   "start" -> si tengono quelle che iniziano dentro la fase (conteggi coerenti con la Fig. 8)
LABEL_RULE = "start"
# su quali finestre di ogni soggetto si calcolano media e std dello z-score:
#   "phases" -> le finestre delle 4 fasi
#   "experiment" -> dalla prima all'ultima annotation;
#   "recording" -> tutta la registrazione
ZSCORE_BASE = "phases"
# assi dell'accelerometro da invertire per i soggetti col braccialetto ruotato di 180° attorno a X.
# Vuoto = dati come pubblicati (riproduzione del paper); correzione: {"ddca342b": ["accY", "accZ"]}
FLIPPED_AXES = {"ddca342b": ["accY", "accZ"]}
#FLIPPED_AXES = {}

WINDOW = pd.Timedelta("30s")  # "resampled to have a sampling frequency of 1/30 seconds" (sez. 3.6)

# (Start, Stop) di ogni fase in *_annotation.csv; "public" è solo il discorso, senza preparazione
PHASE_LABELS = {
    "baseline": ("Baseline Start (Start of Experiment)", "Baseline Stop"),
    "cognitive": ("Cognitive: Start", "Cognitive: Stop"),
    "physical": ("Physical: Start", "Physical: Stop"),
    "public": ("Public Speaking Start", "Public Speaking Stop"),
}

# file -> segnale vitale (già processato da Corsano), da ricampionare a 30 s
VITALS = {"activity.csv": "bpm", "emography.csv": "stress_skin", "temperature.csv": "temp_sk1"}
ZERO_IS_MISSING = {"bpm", "temp_sk1"}  # qui uno 0 indica un valore mancante

PHYSIO = list(VITALS.values())
MOTION = [f"acc{axis}_{stat}" for axis in "XYZ" for stat in ("mean", "std", "var")]
FEATURE_SETS = {"All": PHYSIO + MOTION, "Physio": PHYSIO, "Motion": MOTION}

# fase -> classe, per i due compiti del paper (sez. 5)
TASKS = {
    "binary": {"baseline": "baseline", "physical": "stress", "cognitive": "stress", "public": "stress"},
    "ternary": {"baseline": "baseline", "physical": "physical",
                "cognitive": "cognitive+public", "public": "cognitive+public"},
}

# i 6 classificatori del paper (sez. 3.6) con la griglia di iperparametri della grid search
# (il paper non riporta le griglie: queste sono piccole e scelte da noi)
CLASSIFIERS = {
    "RF": (RandomForestClassifier(random_state=42), {"n_estimators": [100, 300], "max_depth": [None, 10]}),
    "LDA": (LinearDiscriminantAnalysis(solver="lsqr"), {"shrinkage": [None, "auto"]}),
    "kNN": (KNeighborsClassifier(), {"n_neighbors": [3, 5, 9, 15]}),
    "DT": (DecisionTreeClassifier(random_state=42), {"max_depth": [3, 5, 10, None]}),
    "LR": (LogisticRegression(max_iter=1000), {"C": [0.01, 0.1, 1, 10]}),
    "XGB": (XGBClassifier(random_state=42, n_jobs=1), {"n_estimators": [100, 300], "max_depth": [3, 6]}),
}


def read_csv(path, columns, time_column="date"):
    df = pd.read_csv(path, usecols=[time_column, *columns])
    df[time_column] = pd.to_datetime(df[time_column], utc=True, format="ISO8601")  # precisione variabile
    return df.dropna(subset=[time_column]).set_index(time_column)


def phase_intervals(annotation_path):
    """{fase: (start, stop)}: ultimo Stop e ultimo Start prima di esso (gestisce i riavvii)."""
    buttons = read_csv(annotation_path, ["Button Name"], "timestamp")["Button Name"]
    intervals = {}
    for phase, (start_label, stop_label) in PHASE_LABELS.items():
        stop = buttons.index[buttons == stop_label].max()
        starts = buttons.index[(buttons == start_label) & (buttons.index < stop)]
        if starts.empty:  # anche quando manca lo Stop (stop = NaT)
            print(f"  {annotation_path.parent.name}: fase {phase} senza Start/Stop")
            continue
        start = starts.max()
        # id_a5e823ad: "Baseline Start" premuto ~1 h prima; le altre baseline durano ~10 min
        if phase == "baseline" and stop - start > pd.Timedelta("12min"):
            start = stop - pd.Timedelta("10min")
        intervals[phase] = (start, stop)
    return intervals, buttons.index.min(), buttons.index.max()


def subject_windows(files):
    """Una riga per finestra da 30 s: accelerometro aggregato + segnali vitali ricampionati."""
    acc = read_csv(files["acc.csv"], ["accX", "accY", "accZ"]).resample(WINDOW).agg(["mean", "std", "var"])
    acc.columns = [f"{col}_{stat}" for col, stat in acc.columns]  # ("accX", "mean") -> "accX_mean"

    columns = [acc]
    for file, name in VITALS.items():
        signal = read_csv(files[file], [name])[name]
        if name in ZERO_IS_MISSING:
            signal = signal.replace(0, np.nan)
        signal = signal.resample(WINDOW).mean()
        if name == "stress_skin":  # 1 valore/min: riempie la finestra vuota tra due valori
            signal = signal.interpolate(limit=1, limit_area="inside")
        columns.append(signal)
    return pd.concat(columns, axis=1, sort=True)  # sorgenti con estensione temporale diversa


def subject_table(subject_dir, label_rule, zscore_base):
    """Finestre di un soggetto che cadono in una fase, z-scorate e con la colonna "phase"."""
    # "<uuid>_temperature.csv" -> "temperature.csv" (non confonde heat_flux_sensor_temperature)
    files = {f.name.split("_", 1)[1]: f for f in subject_dir.iterdir() if "_" in f.name}
    windows = subject_windows(files)
    for subject, axes in FLIPPED_AXES.items():
        if subject in subject_dir.name:  # invertire il segno cambia solo la media, non std e var
            windows[[f"{axis}_mean" for axis in axes]] *= -1
    intervals, first, last = phase_intervals(files["annotation.csv"])

    start_times, end_times = windows.index, windows.index + WINDOW
    phase = pd.Series(np.nan, index=windows.index, dtype=object)
    for name, (start, stop) in intervals.items():
        ends_inside = end_times <= stop if label_rule == "full" else start_times < stop
        phase[(start_times >= start) & ends_inside] = name

    base = {"phases": phase.notna(),
            "experiment": (start_times >= first) & (end_times <= last),
            "recording": np.ones(len(windows), dtype=bool)}[zscore_base]
    z = (windows - windows[base].mean()) / windows[base].std(ddof=0)
    return z[phase.notna()].assign(phase=phase, subject=subject_dir.name)


def build_table(label_rule=LABEL_RULE, zscore_base=ZSCORE_BASE):
    """Tabella di tutti i soggetti: una riga per finestra etichettata."""
    subject_dirs = sorted(d for d in DATA_DIR.iterdir() if d.is_dir())
    tables = (subject_table(d, label_rule, zscore_base) for d in subject_dirs)
    return pd.concat(tables).rename_axis("window_start").reset_index()


def loso_predictions(X, y, groups, classifier, grid):
    """LOSO annidata: per ogni soggetto di test, la grid search sceglie gli iperparametri con
    una LOSO interna sui soli soggetti di training; il modello migliore, riaddestrato su
    tutti i soggetti di training, predice le finestre del soggetto di test."""
    pred = np.empty_like(y)
    for train, test in LeaveOneGroupOut().split(X, y, groups):
        search = GridSearchCV(classifier, grid, scoring="f1_macro", cv=LeaveOneGroupOut(), n_jobs=-1)
        pred[test] = search.fit(X[train], y[train], groups=groups[train]).predict(X[test])
    return pred


def per_subject_scores(y, pred, groups):
    """Accuracy e macro-F1 di ogni soggetto, cioè di ogni fold della LOSO."""
    df = pd.DataFrame({"y": y, "pred": pred, "subject": groups})
    return df.groupby("subject").apply(lambda d: pd.Series({
        "acc": accuracy_score(d["y"], d["pred"]), "f1": f1_score(d["y"], d["pred"], average="macro")}))


def evaluate_all(table):
    """Una riga per ogni (compito, feature set, classificatore) con media e std sui fold."""
    # LDA, kNN e LR non accettano NaN: si scartano le finestre con almeno una feature mancante
    table = table.dropna(subset=FEATURE_SETS["All"]).reset_index(drop=True)
    rows = []
    for task, mapping in TASKS.items():
        y = table["phase"].map(mapping).factorize(sort=True)[0]  # classi -> 0..n-1 (per XGBoost)
        for set_name, features in FEATURE_SETS.items():
            X = table[features].to_numpy()
            for clf_name, (classifier, grid) in CLASSIFIERS.items():
                groups = table["subject"].to_numpy()
                scores = per_subject_scores(y, loso_predictions(X, y, groups, classifier, grid), groups)
                rows.append({"task": task, "features": set_name, "classifier": clf_name,
                             "acc": scores["acc"].mean(), "acc_std": scores["acc"].std(ddof=0),
                             "f1": scores["f1"].mean(), "f1_std": scores["f1"].std(ddof=0)})
    return pd.DataFrame(rows)


if __name__ == "__main__":
    table = build_table()
    OUT_DIR.mkdir(exist_ok=True)
    out_path = OUT_DIR / f"windows_{LABEL_RULE}_{ZSCORE_BASE}.csv"
    table.to_csv(out_path, index=False)
    print(f"salvata {out_path.name}: {len(table)} finestre, per fase:", table.groupby("phase").size().to_dict())
    print(f"senza NaN: {len(table.dropna(subset=FEATURE_SETS['All']))} finestre "
          "(paper, Fig. 8: 357 baseline + 757 stress)")

    results = evaluate_all(table)
    for metric in ("acc", "f1"):
        results[f"{metric}_text"] = results[metric].map("{:.2f}".format) + " ± " + results[f"{metric}_std"].map("{:.2f}".format)

    # stesso formato delle Tabelle 2-5 del paper: righe = feature set, colonne = classificatori
    for task in TASKS:
        for metric in ("acc", "f1"):
            print(f"\n{task} - {metric}")
            print(results[results["task"] == task].pivot(index="features", columns="classifier", values=f"{metric}_text")
                  .loc[list(FEATURE_SETS), list(CLASSIFIERS)].to_string())


  id_7bb4dafd-5a92-4aef-91e1-d634b40bc353: fase cognitive senza Start/Stop
salvata windows_start_phases.csv: 1196 finestre, per fase: {'baseline': 423, 'cognitive': 206, 'physical': 352, 'public': 215}
senza NaN: 1171 finestre (paper, Fig. 8: 357 baseline + 757 stress)

binary - acc
classifier           RF          LDA          kNN           DT           LR          XGB
features                                                                                
All         0.97 ± 0.04  0.95 ± 0.06  0.94 ± 0.07  0.97 ± 0.04  0.95 ± 0.07  0.98 ± 0.03
Physio      0.94 ± 0.08  0.91 ± 0.10  0.92 ± 0.11  0.94 ± 0.07  0.94 ± 0.07  0.92 ± 0.09
Motion      0.95 ± 0.05  0.92 ± 0.08  0.91 ± 0.10  0.92 ± 0.09  0.90 ± 0.11  0.95 ± 0.05

binary - f1
classifier           RF          LDA          kNN           DT           LR          XGB
features                                                                                
All         0.97 ± 0.05  0.95 ± 0.07  0.94 ± 0.07  0.97 ± 0.04  0.95 ± 0.08  0.9